### 지금까지 배웠던 개념
- 상태(state): 지금 어디에 있고, 어떤 상황인지. 예) 미로의 2행 3열
- 행동(action): 에이전트가 할 수 있는 선택(일). 예) 위/아래/왼쪽/오른쪽
- 보상(Reward): 행동 후 환경이 주는 숫자 피드백. 예) 출구 + 10, 충돌 - 1
- 정책(Policy): 상태를 보고 어떤 행동을 할지 정하는 규칙
- 가치(Value): 당장의 보상뿐 아니라 앞으로 얻을 보상까지 고려한 예상 점수
- 에피소드(Episode): 시작부터 종료까지 한 번의 시도


In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


# **1. TD Learning**

- 어떤 길이 좋은지 판단하려면 원래는 끝까지 가서 받은 보상을 알아야 함
- 미로가 길거나 게임이 오래 지속되면 기다리는 동안 배울 수 없음
- TD는 한 칸 움직인 직후 예상치를 고침
 
```
1. A위치의 미래 점수를 지금 3점으로 예상
2, A에서 B로 이동(즉시 1점을 받음)
3. B에서 앞으로 얻을 점수는 5점이라고 예상
4. 미래 점수를 90%만 반영한다면 새로운 목표는 1 + 0.9 * 5 = 5.5점
5. 현재 예산 3점과 새 목표 5.5점의 차이는 +2.5점, 한 번의 경험만 믿고 3점을 곧바로 5.
5점으로 바꾸면 불안정할 수 있으므로 차의 10%인 0.25점만 반영해  3.25점으로 수정
```

$
delta = r + \gamma V(s') - V(s)
$
- V(s): 이동 전 상태 A에 대해 원래 예상했던 점수 예) 3
- r: 실제 이동 직후 받은 보상 예) 1
- V(s'): 이동 후 상태 B에서 앞으로 받을 것으로 예상하는 점수 예) 5
- $\gamma$: 미래 점수를 현재에 얼마나 반영할지 정하는 할인율 예) 0.8
- $\delta$: 예상과 새목표의 차이. 예) TD 오차 > 2.5

> 5.5점은 미래의 실제 정답이 아니라 현재의 추정값을 사용해 만든 임시 학습 목표. 이를 부트스트래핑(bootstrapping)이라고 부름

In [ ]:
v_a, v_b = 3.0, 5.0
# alpha: 학습률. 새로운 경험을 얼마나 크게 반영할지
reward, gamma, alpha = 1.0, 0.9, 0.1
target = reward + gamma * v_b
td_error = target - v_a
new_v_a = v_a + alpha * td_error
print(f'목표={target:.2f}, 오차={td_error:.2f}, 수정된 V(A)={new_v_a:.2f}')
assert abs(new_v_a - 3.25) < 1e-9

목표=5.50, 오차=2.50, 수정된 V(A)=3.25


1. Monte Carlo
- 업데이트: 에피소드가 끝난 뒤
- 목표: 실제 관찰된 누적 보상
- 장점: 목표에 부트스트래핑 추정값을 사용하지 않음
- 주의점: 종료까지 기다려야 하고, 결과 반동이 클 수 있음

2. TD(Temporal Difference)
- 업데이트: 매 전이 직후
- 목표: 보상 + 다음 상태의 추정 가치
- 장점: 긴 에피소드에도 빠르게 갱싱
- 주의점: 추정값을 다시 사용해 편향이 생길 수 있음

> 둘 중 하나가 우월한 것은 아님. 환경과 학습 조건에 따라 선택

### Monte Carlo(MC)와 TD Learning의 학습 과정 및 성능 비교
1. 0번부터 6번까지 총 7개의 상태로 구성된 1차원 환경을 구현
- 로봇은 항상 3번 상태에서 시작
- 매번 왼쪽 또는 오른쪽으로 50% 확률로 이동
- 0번에 도착하면 보상 0점을 받고 종료
- 6번에 도착하면 보상 +1점을 받고 종료
- 그 외의 이동에서는 보상이 0점

2. Monte Carlo 학습 구현
- 에피소드가 종료된 후 학습
- 실제로 얻은 누적 보상을 계산
- 학습률을 적용하여 상태 가치를 업데이트

3. TD Learning 구현
- 다음 상태의 추정 가치를 이용
- TD Target과 TD Error를 계산
- 학습률을 적용하여 상태 가치를 업데이트

4. 학습 결과 비교
- 에피소드 수: 300
- 학습률: 0.05
- 할인률: 1.0
- 모든 상태 가치의 초기값: 0

5. 학습이 완료되면 아래 내용을 출력
- 각 상태의 실제 가치
- MC가 학습한 상태 가치
- TD가 학습한 상태 가치
- 에피소드별 MC와 TD의 RMSE 변화 그래프

6. 아래 질문에 답하세요.
- MC와 TD Learning은 상태 가치를 업데이트하는 시점과 방법에서 어떤 차이가 있나요?
- RMSE가 작다는 것은 무엇을 의미하나요?
- 실습 결과에서 MC와 TD 중 어느 알고리즘이 더 빠르게 참값에 가까워졌나요? 그 이유는?

In [ ]:
# Random Walk 에피소드 생성
def random_walk_episode(rng):
    state = 3  # 시작 상태
    transitions = []  # 상태 전이 기록

    while state not in (0, 6):  
        action = rng.choice([-1, 1])  # 왼쪽(-1) 또는 오른쪽(1) 무작위 선택
        nxt = state + action  # 다음 상태 계산
        done = nxt in (0, 6)  # 종료 상태 도달 여부 확인
        reward = 1.0 if nxt == 6 else 0.0  # 상태 6 도달 시 보상 1, 그 외 0

        transitions.append((state, reward, nxt, done)) 
        state = nxt  # 현재 상태 갱신

    return transitions  # 에피소드의 전체 전이 기록 반환

# 6에 도착할 확률은 0 1 2 3 4 5 6 에서 3의 위치에 있고 거기서 0과 6의 좌우대칭 
# 즉 중간에 있기 때문에 0에 도착할 확률과 6으로 도착할 확률 반이기 때문에 50%
# 쉽게말하면 0~6의 중간에 있어서 50%임
# 만약 2에 있다면 6에 먼저 도착할 확률이 33%

![오른쪽 도착 확률](./images/오른쪽_도착_확률.png)


![왼쪽 도착 확률](./images/왼쪽_도착_확률.png)

In [ ]:
# alpha: 학습률. 새로운 목표가 나타났다고 기존 값들 한번에 바꾸지 않는다는 뜻
# 새로운 정보와 기존 추정치의 차이를 0.05 반영
def compare_mc_td(episodes=300, alpha=0.05):
    rng = np.random.default_rng(SEED)
    mc = np.zeros(7)
    td = np.zeros(7)
    mc_errors, td_errors = [], []
    truth = np.arange(1, 6) / 6
    for _ in range(episodes):
        path = random_walk_episode(rng)
        for s, r, nxt, done in path:
            td[s] += alpha * (r + (0 if done else td[nxt]) - td[s])
        G = 0.0
        for s, r, nxt, done in reversed(path):
            G = r + G
            mc[s] += alpha * (G - mc[s])
        mc_errors.append(np.sqrt(np.mean((mc[1:6]-truth)**2)))
        td_errors.append(np.sqrt(np.mean((td[1:6]-truth)**2)))
    return mc, td, mc_errors, td_errors